# BÀI TẬP LỚN MÔN NHẬN DẠNG MẪU - ĐỀ TÀI 1: NHẬN DẠNG CHỮ SỐ VIẾT TAY
## NOTEBOOK 1: PHÂN TÍCH KHÁM PHÁ DỮ LIỆU (EDA) & TRÍCH XUẤT ĐẶC TRƯNG
**Sinh viên thực hiện**: Nhóm 03
**Bộ dữ liệu**: MNIST (Modified National Institute of Standards and Technology)
---
### Mục tiêu nghiên cứu:
1. Phân tích cấu trúc dữ liệu, độ cân bằng giữa 10 lớp chữ số ($0 \to 9$).
2. Trực quan hóa ảnh trung bình (*Mean Digit*) và ảnh phương sai (*Variance Digit*).
3. Giảm chiều dữ liệu và khám phá cụm tiềm ẩn bằng **PCA** và **t-SNE**.
4. Đánh giá tác động của kỹ thuật nắn thẳng góc nghiêng (*Deskewing*) và trích xuất đặc trưng *HOG*.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thêm thư mục gốc vào path
sys.path.append(str(Path.cwd().parent))
from src.data_loader import load_mnist_data
from src.features import deskew_image, extract_hog_features, PCATransformer

print('Đã nạp thành công các thư viện cốt lõi!')


### 1. Nạp dữ liệu và kiểm tra kích thước


In [ ]:
(x_train, y_train), (x_test, y_test) = load_mnist_data(normalize=True, flatten=False)
print(f'Tập huấn luyện (Train): {x_train.shape}, Nhãn: {y_train.shape}')
print(f'Tập kiểm thử   (Test) : {x_test.shape},  Nhãn: {y_test.shape}')
print(f'Miền giá trị pixel    : [{x_train.min():.1f}, {x_train.max():.1f}]')


### 2. Kiểm tra độ cân bằng giữa 10 lớp chữ số (Class Distribution)


In [ ]:
train_counts = np.bincount(y_train, minlength=10)
test_counts = np.bincount(y_test, minlength=10)

plt.figure(figsize=(10, 4), dpi=150)
x = np.arange(10)
plt.bar(x - 0.2, train_counts, width=0.4, label='Train (60,000)', color='#1d3557')
plt.bar(x + 0.2, test_counts, width=0.4, label='Test (10,000)', color='#e63946')
plt.xlabel('Chữ số (0 - 9)')
plt.ylabel('Số lượng mẫu')
plt.title('Phân bố số lượng mẫu giữa các lớp MNIST')
plt.xticks(x)
plt.legend()
plt.show()


### 3. Trực quan hóa ảnh trung bình và phương sai của từng chữ số
- **Mean Digit Image**: Cho thấy hình dạng chuẩn mực, trung tâm của từng chữ số.
- **Variance Image**: Vùng phát sáng mạnh đại diện cho những vị trí có sự biến thiên nét bút lớn nhất.


In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 3.5), dpi=150)
for d in range(10):
    imgs = x_train[y_train == d]
    axes[0, d].imshow(np.mean(imgs, axis=0), cmap='viridis')
    axes[0, d].set_title(f'Mean {d}')
    axes[0, d].axis('off')
    
    axes[1, d].imshow(np.var(imgs, axis=0), cmap='magma')
    axes[1, d].set_title(f'Var {d}')
    axes[1, d].axis('off')
plt.tight_layout()
plt.show()


### 4. Kỹ thuật tiền xử lý hình học: Nắn thẳng góc nghiêng (Deskewing)
Sử dụng mô-men ảnh trung tâm để tính độ nghiêng nét viết và khử góc nghiêng bằng phép biến đổi Affine.


In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(12, 4), dpi=150)
for i in range(6):
    orig = x_train[i]
    desk = deskew_image(orig)
    axes[0, i].imshow(orig, cmap='gray')
    axes[0, i].set_title(f'Gốc (Lớp: {y_train[i]})')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(desk, cmap='gray')
    axes[1, i].set_title('Sau nắn thẳng')
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()


### 5. Phân tích thành phần chính (PCA) & Không gian tiềm ẩn t-SNE
Khảo sát phương sai tích lũy và tìm số chiều tối ưu để giữ 95% thông tin.


In [ ]:
from sklearn.decomposition import PCA
X_flat = x_train.reshape(len(x_train), -1)
pca = PCA(n_components=150, random_state=42)
pca.fit(X_flat[:10000])
cum_var = np.cumsum(pca.explained_variance_ratio_)
n_95 = np.searchsorted(cum_var, 0.95) + 1

plt.figure(figsize=(7, 4), dpi=150)
plt.plot(range(1, 151), cum_var * 100, color='#0284c7', lw=2)
plt.axhline(95, color='r', linestyle='--', label=f'Ngưỡng 95% ({n_95} chiều)')
plt.xlabel('Số thành phần chính (PC)')
plt.ylabel('Phương sai tích lũy (%)')
plt.title('Đường cong phương sai tích lũy của PCA')
plt.legend()
plt.show()
